In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from torch.nn import functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from domain_shift.core.config import settings
from domain_shift.CycleGAN.triplet_data_loader_val import get_data_loader
from domain_shift.CycleGAN.triplet_models_val import CycleGAN
from domain_shift.data_extraction.process_DRIAMS import DRIAMS_bin_to_df

In [5]:
cycle_gan = CycleGAN()
cycle_gan.load_checkpoint_models()

/home/dive001/Documents/Master/maldi-tof-domain-shift/domain_shift/CycleGAN/triplet_models_val.py:650: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(settings.TEMP

In [ ]:
# Load the data
driams_1 = DRIAMS_bin_to_df(settings.DRIAMS_C_PATH)
driams_2 = DRIAMS_bin_to_df(settings.DRIAMS_D_PATH)
driams_3 = DRIAMS_bin_to_df(settings.DRIAMS_B_PATH)

In [7]:
# Combine the 'species' columns from both datasets
combined_species = pd.concat([driams_1['species'], driams_2['species']])

# Find the most represented species across both datasets
species_counts = combined_species.value_counts()
top_species = species_counts.index[:2]
top_species = top_species.tolist()
top_species

['Escherichia coli', 'Staphylococcus aureus']

In [8]:
# Show the distribution of the most represented species in each dataset
driams_1['species'].value_counts()[top_species], \
driams_2['species'].value_counts()[top_species]

(species
 Escherichia coli         927
 Staphylococcus aureus    738
 Name: count, dtype: int64,
 species
 Escherichia coli         2013
 Staphylococcus aureus    2174
 Name: count, dtype: int64)

In [9]:
# Filter by the 4 species with most representation
filtered_driams_1 = driams_1[driams_1["species"].isin(top_species)]
filtered_driams_2 = driams_2[driams_2["species"].isin(top_species)]
filtered_driams_3 = driams_3[driams_3["species"].isin(top_species)]

In [12]:
filtered_driams_1['species'].value_counts(), \
filtered_driams_2['species'].value_counts(), \
filtered_driams_3['species'].value_counts()

(species
 Escherichia coli         927
 Staphylococcus aureus    738
 Name: count, dtype: int64,
 species
 Staphylococcus aureus    2174
 Escherichia coli         2013
 Name: count, dtype: int64,
 species
 Staphylococcus aureus    353
 Escherichia coli         213
 Name: count, dtype: int64)

In [13]:
# Generate the synthetic data
synthetic_driams_1 = []
for index, data_anchor in filtered_driams_1.iterrows():
    binned_data = data_anchor["binned_6000"]

    # Convert to tensor if necessary
    if not isinstance(binned_data, torch.Tensor):
        binned_data = torch.tensor(binned_data, dtype=torch.float32)

    # Generate synthetic data
    output = cycle_gan.generator_2_to_1(binned_data.unsqueeze(0).to("cuda")).cpu().detach()
    
    # Flatten properly and append
    synthetic_driams_1.append(output.view(output.size(0), -1))  

# Stack tensors
synthetic_driams_1 = torch.cat(synthetic_driams_1)

# Test the model on the synthetic driams 2
X = synthetic_driams_1.numpy()
y = filtered_driams_1["species"].values

In [14]:
# Generate the synthetic data
synthetic_driams_2 = []
for index, data_anchor in filtered_driams_2.iterrows():
    binned_data = data_anchor["binned_6000"]

    # Convert to tensor if necessary
    if not isinstance(binned_data, torch.Tensor):
        binned_data = torch.tensor(binned_data, dtype=torch.float32)

    # Generate synthetic data
    output = cycle_gan.generator_2_to_1(binned_data.unsqueeze(0).to("cuda")).cpu().detach()
    
    # Flatten properly and append
    synthetic_driams_2.append(output.view(output.size(0), -1))  

# Stack tensors
synthetic_driams_2 = torch.cat(synthetic_driams_2)

# Test the model on the synthetic driams 2
X = synthetic_driams_2.numpy()
y = filtered_driams_2["species"].values

In [15]:
# Generate the synthetic data
synthetic_driams_3 = []
for index, data_anchor in filtered_driams_3.iterrows():
    binned_data = data_anchor["binned_6000"]

    # Convert to tensor if necessary
    if not isinstance(binned_data, torch.Tensor):
        binned_data = torch.tensor(binned_data, dtype=torch.float32)

    # Generate synthetic data
    output = cycle_gan.generator_2_to_1(binned_data.unsqueeze(0).to("cuda")).cpu().detach()
    
    # Flatten properly and append
    synthetic_driams_3.append(output.view(output.size(0), -1))  

# Stack tensors
synthetic_driams_3 = torch.cat(synthetic_driams_3)

# Test the model on the synthetic driams 2
X = synthetic_driams_3.numpy()
y = filtered_driams_3["species"].values

In [16]:
# Train Random Forest on DRIAMS 1
X = synthetic_driams_1
y = filtered_driams_1["species"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

f1_score(y_test, y_pred, average='weighted')

1.0

In [17]:
# Test the model on the filtered driams 2
X = np.vstack(filtered_driams_2["binned_6000"].values)
y = filtered_driams_2["species"].values

y_pred = rf.predict(X)

f1_score(y, y_pred, average='weighted')

0.3549120286712524

In [18]:
# Test the model on the filtered driams 2
X = synthetic_driams_2
y = filtered_driams_2["species"].values

y_pred = rf.predict(X)

f1_score(y, y_pred, average='weighted')

0.9116456692893575

In [19]:
# Test the model on the filtered driams 3
X = np.vstack(filtered_driams_3["binned_6000"].values)
y = filtered_driams_3["species"].values

y_pred = rf.predict(X)

f1_score(y, y_pred, average='weighted')

0.4791234903509346

In [20]:
# Test the model on the filtered driams 3
X = synthetic_driams_3
y = filtered_driams_3["species"].values

y_pred = rf.predict(X)

f1_score(y, y_pred, average='weighted')

0.9982323864717142